In [1]:
# Setup: Copy files, fix names, set paths

import os
import sys
import shutil

# ── Config ────────────────────────────────────────────────────
SRC_ROOT  = '/kaggle/input/datasets/krishforai/repocoder-stage4/repocoder_stage4'
DST_ROOT  = '/kaggle/working/repocoder_stage4'
SRC_PATH  = os.path.join(DST_ROOT, 'src')
REPO_PATH = os.path.join(DST_ROOT, 'sample_repo')
OUT_DIR   = os.path.join(DST_ROOT, 'output')
EMB_DIR   = os.path.join(DST_ROOT, 'embeddings')

# ── Copy project (always fresh to avoid stale files) ─────────
if os.path.exists(DST_ROOT):
    shutil.rmtree(DST_ROOT)
shutil.copytree(SRC_ROOT, DST_ROOT)
print("✅ Project copied to working directory")

# ── Fix capitalised filenames ─────────────────────────────────
renames = {
    'Preprocessor.py': 'preprocessor.py',
    'Dataloader.py'  : 'data_loader.py',
}
utils_dir = os.path.join(REPO_PATH, 'utils')
for wrong, correct in renames.items():
    wrong_path   = os.path.join(utils_dir, wrong)
    correct_path = os.path.join(utils_dir, correct)
    if os.path.exists(wrong_path):
        os.rename(wrong_path, correct_path)
        print(f"✅ Renamed {wrong} → {correct}")
    else:
        print(f"ℹ️  {correct} already correct")

# ── Create output folders ─────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)
print("✅ Output folders ready")

# ── Set working directory and Python path ────────────────────
os.chdir(DST_ROOT)
for p in [DST_ROOT, SRC_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Setup complete")

✅ Project copied to working directory
✅ Renamed Preprocessor.py → preprocessor.py
✅ Renamed Dataloader.py → data_loader.py
✅ Output folders ready
✅ Working directory: /kaggle/working/repocoder_stage4
✅ Setup complete


In [2]:
# Install dependencies (skips if already installed)

import importlib

def is_installed(package):
    try:
        importlib.import_module(package)
        return True
    except ImportError:
        return False

to_install = []
if not is_installed('sentence_transformers'):
    to_install.append('sentence-transformers')
if not is_installed('faiss'):
    to_install.append('faiss-cpu')

if to_install:
    print(f"Installing: {to_install}")
    os.system(f"pip install {' '.join(to_install)} -q")
    print("✅ Installation complete")
else:
    print("✅ All dependencies already installed — skipping")

Installing: ['faiss-cpu']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.9 MB/s eta 0:00:00
✅ Installation complete


In [3]:
# Verify all imports

import numpy as np
import faiss
import networkx as nx
import matplotlib
from sentence_transformers import SentenceTransformer

print(f"✅ numpy               {np.__version__}")
print(f"✅ faiss               OK")
print(f"✅ networkx            {nx.__version__}")
print(f"✅ matplotlib          {matplotlib.__version__}")
print(f"✅ sentence-transformers OK")
print(f"\n✅ All imports verified — ready to run pipeline")

✅ numpy               2.4.6
✅ faiss               OK
✅ networkx            3.6.1
✅ matplotlib          3.10.0
✅ sentence-transformers OK

✅ All imports verified — ready to run pipeline


In [4]:
# Step 1: AST Parser

import json
from step1_ast_parser import RepositoryIndexer

INDEX_FILE = os.path.join(OUT_DIR, 'repo_index.json')

# Skip if already done
if os.path.exists(INDEX_FILE):
    print("ℹ️  repo_index.json already exists — loading from cache")
    with open(INDEX_FILE) as f:
        cached = json.load(f)
    print(f"   Cached stats: {cached['stats']}")
    # Still need indexer object for downstream steps
    indexer = RepositoryIndexer(repo_path=REPO_PATH, output_dir=OUT_DIR)
    indexer.index_repository()
else:
    indexer = RepositoryIndexer(repo_path=REPO_PATH, output_dir=OUT_DIR)
    stats = indexer.index_repository()
    print(f"\n✅ Step 1 complete")
    print(f"   {stats}")


STAGE 4 - STEP 1: Repository Indexing
Repository: /kaggle/working/repocoder_stage4/sample_repo

Found 5 Python files

  Parsing: __init__.py
  Parsing: models/code_models.py
  Parsing: services/evaluation.py
  Parsing: utils/data_loader.py
  Parsing: utils/preprocessor.py

────────────────────────────────────────
INDEX SUMMARY
────────────────────────────────────────
  total_files              : 5
  parsed_files             : 5
  parse_errors             : 0
  total_functions          : 16
  total_classes            : 5
  total_loc                : 586

Index saved → /kaggle/working/repocoder_stage4/output/repo_index.json

✅ Step 1 complete
   {'total_files': 5, 'parsed_files': 5, 'parse_errors': 0, 'total_functions': 16, 'total_classes': 5, 'total_loc': 586}


In [5]:
# Step 2: Embeddings

import numpy as np
from step2_embeddings import CodeEmbedder, EmbeddingBuilder

EMB_FILE = os.path.join(EMB_DIR, 'function_embeddings.npy')

# Load embedder always (needed downstream even if embeddings cached)
embedder = CodeEmbedder(model_name='all-MiniLM-L6-v2')

if os.path.exists(EMB_FILE):
    print("ℹ️  Embeddings already exist — skipping rebuild")
    existing = np.load(EMB_FILE)
    print(f"   Cached function embeddings shape: {existing.shape}")
    print(f"✅ Step 2 complete (from cache)")
else:
    print("Building embeddings from scratch...")
    builder = EmbeddingBuilder(
        indexer=indexer,
        embedder=embedder,
        output_dir=EMB_DIR
    )
    summary = builder.build_all()
    print(f"\n✅ Step 2 complete")
    print(f"   {summary}")


Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


  Saved embeddings → /kaggle/working/repocoder_stage4/embeddings/function_embeddings.npy  shape=(16, 384)
  Saved metadata   → /kaggle/working/repocoder_stage4/embeddings/function_metadata.json

────────────────────────────────────────
Building Class Embeddings
────────────────────────────────────────
Total classes to embed: 5
  Encoding 5 texts in batches of 16...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Saved embeddings → /kaggle/working/repocoder_stage4/embeddings/class_embeddings.npy  shape=(5, 384)
  Saved metadata   → /kaggle/working/repocoder_stage4/embeddings/class_metadata.json

────────────────────────────────────────
Building Module (File) Embeddings
────────────────────────────────────────
Total modules to embed: 5
  Encoding 5 texts in batches of 16...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Saved embeddings → /kaggle/working/repocoder_stage4/embeddings/module_embeddings.npy  shape=(5, 384)
  Saved metadata   → /kaggle/working/repocoder_stage4/embeddings/module_metadata.json

EMBEDDING SUMMARY
  function_embeddings: {'count': 16, 'shape': [16, 384]}
  class_embeddings: {'count': 5, 'shape': [5, 384]}
  module_embeddings: {'count': 5, 'shape': [5, 384]}
  embedding_model: all-MiniLM-L6-v2
  embedding_dim: 384

✅ Step 2 complete. Proceed to step3_faiss_index.py

✅ Step 2 complete
   {'function_embeddings': {'count': 16, 'shape': [16, 384]}, 'class_embeddings': {'count': 5, 'shape': [5, 384]}, 'module_embeddings': {'count': 5, 'shape': [5, 384]}, 'embedding_model': 'all-MiniLM-L6-v2', 'embedding_dim': 384}


In [6]:
# Step 3: FAISS Index + Search Test

from step3_faiss_index import FAISSIndexBuilder, CodeSearchEngine

FAISS_FILE = os.path.join(EMB_DIR, 'function_index.faiss')

faiss_builder = FAISSIndexBuilder(embedding_dir=EMB_DIR)

if os.path.exists(FAISS_FILE):
    print("ℹ️  FAISS index already exists — loading from cache")
    faiss_builder.load_existing_indices()
    print(f"   Loaded {faiss_builder.indices['function'].ntotal} function vectors")
else:
    print("Building FAISS indices from scratch...")
    faiss_builder.build_all_indices()

# Search engine always needs to be created fresh (not cached to disk)
engine = CodeSearchEngine(
    index_builder=faiss_builder,
    embedder=embedder
)

# Quick search test
print("\n--- Search test: 'load data from csv file' ---")
results = engine.search("load data from csv file", top_k=3)
engine.display_results("load data from csv file", results)
print(f"\n✅ Step 3 complete")

Building FAISS indices from scratch...

STAGE 4 - STEP 3: FAISS Index Construction

Building function index...
  ✅ function index: IndexFlatIP (exact)
     Vectors=16, Dim=384, Saved→/kaggle/working/repocoder_stage4/embeddings/function_index.faiss

Building class index...
  ✅ class index: IndexFlatIP (exact)
     Vectors=5, Dim=384, Saved→/kaggle/working/repocoder_stage4/embeddings/class_index.faiss

Building module index...
  ✅ module index: IndexFlatIP (exact)
     Vectors=5, Dim=384, Saved→/kaggle/working/repocoder_stage4/embeddings/module_index.faiss

✅ Step 3 complete. Proceed to step4_repository_explorer.py

--- Search test: 'load data from csv file' ---
  Encoding 1 texts in batches of 32...

SEARCH QUERY: "load data from csv file"
  #1  [FUNCTION]  score=0.5861
       Name    : load_csv
       File    : /kaggle/working/repocoder_stage4/sample_repo/utils/data_loader.py:10
       Sig     : def load_csv(filepath, delimiter)
       Doc     : Load a CSV file and return a list of dic

In [7]:
# Step 4: Dependency Tracer

from step4_dependency_tracer import DependencyTracer, GraphVisualizer

DEP_FILE  = os.path.join(OUT_DIR, 'dependency_graph.json')
CALL_FILE = os.path.join(OUT_DIR, 'call_graph.json')

tracer = DependencyTracer(indexer=indexer)

if os.path.exists(DEP_FILE) and os.path.exists(CALL_FILE):
    print("ℹ️  Dependency graphs already exist — loading from cache")
    with open(DEP_FILE) as f:
        import_graph = json.load(f)
    with open(CALL_FILE) as f:
        call_graph = json.load(f)
    # Still rebuild in-memory graph for viz (not serialised to disk)
    tracer.build_import_graph()
    tracer.build_call_graph()
    print(f"   Import edges: {sum(len(v) for v in import_graph.values())}")
    print(f"   Call edges  : {sum(len(v) for v in call_graph.values())}")
else:
    print("Building dependency graphs from scratch...")
    import_graph = tracer.build_import_graph()
    call_graph   = tracer.build_call_graph()
    tracer.save_graphs(output_dir=OUT_DIR)
    print(f"   Import edges: {sum(len(v) for v in import_graph.values())}")
    print(f"   Call edges  : {sum(len(v) for v in call_graph.values())}")

# Dependency report
viz = GraphVisualizer(tracer=tracer, output_dir=OUT_DIR)
viz.print_dependency_report(repo_path=REPO_PATH)

# Visualisation
PNG_FILE = os.path.join(OUT_DIR, 'dependency_graph.png')
if os.path.exists(PNG_FILE):
    print("\nℹ️  dependency_graph.png already exists — displaying cached version")
    from IPython.display import Image, display
    display(Image(PNG_FILE))
else:
    result = viz.visualize_import_graph(repo_path=REPO_PATH)
    if result and os.path.exists(PNG_FILE):
        from IPython.display import Image, display
        display(Image(PNG_FILE))
    else:
        print("ℹ️  No intra-repo import edges — PNG skipped (expected for sample_repo)")

print(f"\n✅ Step 4 complete")


Building dependency graphs from scratch...
  Saved dependency graph → /kaggle/working/repocoder_stage4/output/dependency_graph.json
  Saved call graph       → /kaggle/working/repocoder_stage4/output/call_graph.json
   Import edges: 0
   Call edges  : 4

DEPENDENCY ANALYSIS REPORT
  No intra-repo dependencies detected.
  (All imports are external packages)
  ⚠ No dependency edges found to visualize.
ℹ️  No intra-repo import edges — PNG skipped (expected for sample_repo)

✅ Step 4 complete


In [8]:
# Step 5: Repository Explorer

from step5_repository_explorer import RepositoryExplorer

# rebuild=False because Steps 1-4 already built all artifacts
explorer = RepositoryExplorer(
    repo_path=REPO_PATH,
    output_dir=OUT_DIR,
    embedding_dir=EMB_DIR,
    model_name='all-MiniLM-L6-v2',
    rebuild=False
)

print("\n--- SEMANTIC SEARCH ---")
results = explorer.search("normalize text input", top_k=3)
explorer.display_search_results("normalize text input", results)

print("\n--- EXACT LOOKUP ---")
fns = explorer.find_function("load_csv")
if fns:
    explorer.display_function_info(fns[0])
else:
    print("  load_csv not found")

print("\n--- STRUCTURAL QUERY ---")
deps = explorer.get_dependents("utils/data_loader.py")
print(f"  Files importing data_loader: {deps if deps else 'none (expected)'}")

print("\n--- EVALUATION METRICS ---")
metrics = explorer.run_evaluation()
print(f"\n  {metrics}")

print(f"\n✅ Step 5 complete")


REPOSITORY EXPLORER — Stage 4 Final Deliverable
Repository: /kaggle/working/repocoder_stage4/sample_repo


STAGE 4 - STEP 1: Repository Indexing
Repository: /kaggle/working/repocoder_stage4/sample_repo

Found 5 Python files

  Parsing: __init__.py
  Parsing: models/code_models.py
  Parsing: services/evaluation.py
  Parsing: utils/data_loader.py
  Parsing: utils/preprocessor.py

────────────────────────────────────────
INDEX SUMMARY
────────────────────────────────────────
  total_files              : 5
  parsed_files             : 5
  parse_errors             : 0
  total_functions          : 16
  total_classes            : 5
  total_loc                : 586

Index saved → /kaggle/working/repocoder_stage4/output/repo_index.json

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ Model loaded. Embedding dim = 384
  ℹ Using cached embeddings. Pass rebuild=True to regenerate.
  ℹ Loading existing FAISS indices...
  Loaded function index: 16 vectors
  Loaded class index: 5 vectors
  Loaded module index: 5 vectors

✅ Repository Explorer ready!


--- SEMANTIC SEARCH ---
  Encoding 1 texts in batches of 32...

SEARCH QUERY: "normalize text input"
  #1  [FUNCTION]  score=0.6551
       Name    : normalize_text
       File    : /kaggle/working/repocoder_stage4/sample_repo/utils/preprocessor.py:9
       Sig     : def normalize_text(text)
       Doc     : Normalize text by lowercasing and removing extra whitespace.

Args:
    text: Raw input text.

Retur...

  #2  [MODULE]  score=0.4224
       Name    : unknown
       File    : /kaggle/working/repocoder_stage4/sample_repo/utils/preprocessor.py:0
       Doc     : Text and code preprocessing utilities....

  #3  [FUNCTION]  score=0.3599
       Name    : tokenize
       File    : /kaggle/working/repocoder_stage4/sample_r

In [11]:
# Final verification of all artifacts

print("=" * 55)
print("STAGE 4 — FINAL ARTIFACT CHECK")
print("=" * 55)

expected = {
    'AST Index'            : os.path.join(OUT_DIR, 'repo_index.json'),
    'Dependency Graph JSON': os.path.join(OUT_DIR, 'dependency_graph.json'),
    'Call Graph JSON'      : os.path.join(OUT_DIR, 'call_graph.json'),
    'Explorer Report'      : os.path.join(OUT_DIR, 'explorer_report.json'),
    'Function Embeddings'  : os.path.join(EMB_DIR, 'function_embeddings.npy'),
    'Class Embeddings'     : os.path.join(EMB_DIR, 'class_embeddings.npy'),
    'Module Embeddings'    : os.path.join(EMB_DIR, 'module_embeddings.npy'),
    'Function FAISS Index' : os.path.join(EMB_DIR, 'function_index.faiss'),
    'Class FAISS Index'    : os.path.join(EMB_DIR, 'class_index.faiss'),
    'Module FAISS Index'   : os.path.join(EMB_DIR, 'module_index.faiss'),
    'Function Metadata'    : os.path.join(EMB_DIR, 'function_metadata.json'),
    'Class Metadata'       : os.path.join(EMB_DIR, 'class_metadata.json'),
    'Module Metadata'      : os.path.join(EMB_DIR, 'module_metadata.json'),
}

all_ok = True
for name, path in expected.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  ✅  {name:<25} {size:>10,} bytes")
    else:
        print(f"  ❌  {name:<25} MISSING")
        all_ok = False

print("\n" + "=" * 55)
if all_ok:
    print("✅ ALL ARTIFACTS PRESENT — Stage 4 complete")
    print("   Next: Stage 5 RAG pipeline uses this explorer as Retriever")
else:
    print("❌ Some artifacts missing — re-run the failing step cell")
print("=" * 55)

STAGE 4 — FINAL ARTIFACT CHECK
  ✅  AST Index                     31,877 bytes
  ✅  Dependency Graph JSON              2 bytes
  ✅  Call Graph JSON                  554 bytes
  ✅  Explorer Report                1,267 bytes
  ✅  Function Embeddings           24,704 bytes
  ✅  Class Embeddings               7,808 bytes
  ✅  Module Embeddings              7,808 bytes
  ✅  Function FAISS Index          24,621 bytes
  ✅  Class FAISS Index              7,725 bytes
  ✅  Module FAISS Index             7,725 bytes
  ✅  Function Metadata             12,609 bytes
  ✅  Class Metadata                 3,305 bytes
  ✅  Module Metadata                1,950 bytes

✅ ALL ARTIFACTS PRESENT — Stage 4 complete
   Next: Stage 5 RAG pipeline uses this explorer as Retriever


In [10]:
# Fix: generate the missing explorer_report.json
report = explorer.generate_summary_report()

print("Generated report:")
print(f"  Repository : {report['repository']}")
print(f"  Files      : {report['total_files']}")
print(f"  Functions  : {report['total_functions']}")
print(f"  Classes    : {report['total_classes']}")
print(f"  Total LOC  : {report['total_loc']}")
print(f"\n✅ Explorer report generated")

  Summary report saved → /kaggle/working/repocoder_stage4/output/explorer_report.json
Generated report:
  Repository : /kaggle/working/repocoder_stage4/sample_repo
  Files      : 5
  Functions  : 16
  Classes    : 5
  Total LOC  : 586

✅ Explorer report generated
